In [4]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

def process_single_trace_10sec(hdf5_path, metadata_path, target_trace_name=None):
    """
    從 HDF5 與 Metadata 中讀取一筆資料，切割 P 波前後 5 秒。
    執行動作：
    1. 繪製波形圖 (.png)
    2. 儲存原始數值 (.csv 與 .npy)
    """
    
    # 1. 讀取 Metadata
    if not os.path.exists(metadata_path) or not os.path.exists(hdf5_path):
        print("錯誤：找不到 metadata 或 hdf5 檔案，請檢查路徑。")
        return

    df = pd.read_csv(metadata_path)
    
    # 2. 決定要抓哪一筆資料
    if target_trace_name:
        row = df[df["trace_name"] == target_trace_name]
        if row.empty:
            print(f"錯誤：在 Metadata 中找不到 trace_name: {target_trace_name}")
            return
        row = row.iloc[0]
    else:
        # 預設抓第一筆 P arrival > 500 samples 的資料
        valid_rows = df[df["trace_p_arrival_sample"] >= 500]
        if valid_rows.empty:
            print("錯誤：沒有任何資料的 P arrival 大於 5 秒，無法切割。")
            return
        row = valid_rows.iloc[0] 
        
    trace_name = str(row['trace_name'])
    p_arrival = int(row['trace_p_arrival_sample'])
    
    print(f"🎯 目標 Trace: {trace_name}")
    print(f"⏱️ P-arrival index: {p_arrival}")

    # 3. 設定切割參數 (前後 5 秒)
    fs = 100             # 採樣率 100 Hz
    pre_sec = 5          # 前 5 秒
    post_sec = 5         # 後 5 秒
    
    start_idx = p_arrival - (pre_sec * fs)
    end_idx = p_arrival + (post_sec * fs)
    
    print(f"✂️ 切割範圍: {start_idx} ~ {end_idx} (共 {end_idx - start_idx} samples)")

    # 4. 從 HDF5 讀取波形
    waveform_slice = None
    
    with h5py.File(hdf5_path, 'r') as h5:
        # 搜尋對應的 Key
        found_key = next((k for k in h5.keys() if k.endswith(f"_{trace_name}")), None)
        
        if found_key is None:
            print(f"錯誤：在 HDF5 中找不到 Key 結尾為 {trace_name} 的資料")
            return
            
        full_data = h5[found_key][()] 
        
        # 檢查邊界
        if start_idx < 0 or end_idx > full_data.shape[1]:
            print(f"錯誤：數據長度不足。")
            return
            
        # 執行切割 (Shape: 3, 1000)
        waveform_slice = full_data[:, start_idx:end_idx]

    # 5. 執行儲存與繪圖
    if waveform_slice is not None:
        # 建立輸出資料夾
        output_dir = "extracted_data"
        os.makedirs(output_dir, exist_ok=True)

        # A. 儲存原始數值 (Function call)
        save_waveform_data(waveform_slice, trace_name, fs, output_dir)
        
        # B. 繪圖 (Function call)
        plot_waveform(waveform_slice, trace_name, fs, output_dir)

def save_waveform_data(waveform, trace_name, fs, output_dir):
    """
    將波形數值存檔 (.csv 和 .npy)
    waveform shape: (3, N) -> (Z, N, E)
    """
    # 1. 儲存為 .npy (Numpy Binary) - 最適合程式讀取
    npy_path = os.path.join(output_dir, f"{trace_name}_10sec.npy")
    np.save(npy_path, waveform)
    print(f"💾 原始陣列已儲存 (.npy): {npy_path}")

    # 2. 儲存為 .csv (Excel Readable) - 最適合人類查看
    # 轉換維度：(3, 1000) -> (1000, 3) 以便存成表格
    waveform_T = waveform.T 
    time_axis = np.arange(waveform.shape[1]) / fs  # 0.00, 0.01, ... 9.99
    
    # 建立 DataFrame
    df_save = pd.DataFrame({
        "Time_sec": time_axis,
        "Z_axis": waveform_T[:, 0], # 假設 channel 0 是 Z
        "N_axis": waveform_T[:, 1], # 假設 channel 1 是 N
        "E_axis": waveform_T[:, 2]  # 假設 channel 2 是 E
    })
    
    csv_path = os.path.join(output_dir, f"{trace_name}_10sec.csv")
    df_save.to_csv(csv_path, index=False, float_format='%.6f')
    print(f"📝 數值表格已儲存 (.csv): {csv_path}")

def plot_waveform(waveform, trace_name, fs, output_dir):
    """
    繪製並儲存波形圖
    """
    samples = waveform.shape[1]
    t = np.arange(samples) / fs
    
    fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    channel_labels = ["Z", "N", "E"]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] 

    for i in range(3):
        axs[i].plot(t, waveform[i], color=colors[i], linewidth=1)
        axs[i].set_title(f"Channel: {channel_labels[i]}")
        axs[i].set_ylabel("Amplitude")
        axs[i].grid(True, linestyle='--', alpha=0.7)
        # 標示 P 波 (第 5 秒)
        axs[i].axvline(x=5, color='r', linestyle='--', label='P-arrival')
        if i == 0:
            axs[i].legend(loc='upper right')

    axs[-1].set_xlabel("Time (s) [0-10s]")
    plt.suptitle(f"Trace: {trace_name} (10s window)", fontsize=14)
    plt.tight_layout()
    
    png_path = os.path.join(output_dir, f"{trace_name}_waveform.png")
    plt.savefig(png_path)
    print(f"🖼️ 波形圖片已儲存 (.png): {png_path}")
    plt.close() # 關閉圖表釋放記憶體

if __name__ == "__main__":

    metadata_path = r"Y:\CWA_processed_data\all_metadata.csv"
    hdf5_path = r"Y:\CWA_processed_data\all.hdf5"
    
    # 執行處理
    process_single_trace_10sec(hdf5_path, metadata_path)
    
    # 如果要指定某一筆：
    splite_to_wave_list = ["1205202239_HWA_HL_BH_10","1205202239_HWA_HL_SMT_10","1205202239_EHY_HL_SMT_10","1205202239_TWD_HL_SMT_10"]
    for item in splite_to_wave_list:
        process_single_trace_10sec(hdf5_path, metadata_path, target_trace_name=item)

🎯 目標 Trace: 1205202239_EGFH_HL_BH_10
⏱️ P-arrival index: 4802
✂️ 切割範圍: 4302 ~ 5302 (共 1000 samples)
💾 原始陣列已儲存 (.npy): extracted_data\1205202239_EGFH_HL_BH_10_10sec.npy
📝 數值表格已儲存 (.csv): extracted_data\1205202239_EGFH_HL_BH_10_10sec.csv
🖼️ 波形圖片已儲存 (.png): extracted_data\1205202239_EGFH_HL_BH_10_waveform.png
🎯 目標 Trace: 1205202239_HWA_HL_BH_10
⏱️ P-arrival index: 5192
✂️ 切割範圍: 4692 ~ 5692 (共 1000 samples)
💾 原始陣列已儲存 (.npy): extracted_data\1205202239_HWA_HL_BH_10_10sec.npy
📝 數值表格已儲存 (.csv): extracted_data\1205202239_HWA_HL_BH_10_10sec.csv
🖼️ 波形圖片已儲存 (.png): extracted_data\1205202239_HWA_HL_BH_10_waveform.png
🎯 目標 Trace: 1205202239_HWA_HL_SMT_10
⏱️ P-arrival index: 5192
✂️ 切割範圍: 4692 ~ 5692 (共 1000 samples)
💾 原始陣列已儲存 (.npy): extracted_data\1205202239_HWA_HL_SMT_10_10sec.npy
📝 數值表格已儲存 (.csv): extracted_data\1205202239_HWA_HL_SMT_10_10sec.csv
🖼️ 波形圖片已儲存 (.png): extracted_data\1205202239_HWA_HL_SMT_10_waveform.png
🎯 目標 Trace: 1205202239_EHY_HL_SMT_10
⏱️ P-arrival index: 5103
✂️ 切割範圍: 4603 ~ 56

### 每個地震建立資料夾 裡面放該地震事件的所有波形.numpy檔案與.CSV檔案
還沒完成

In [10]:
import os
import pandas as pd
import re
metadata_path = r"Y:\\CWA_processed_data\\all_metadata.csv"
df = pd.read_csv(metadata_path,usecols=["trace_name"])
earthquake_list = []
# 提取地震的編號
extracted_ids = df.str.extract(r'^(\d+)')
uni_ids = extracted_ids[0].dropna().unique()
# 5. 檢查並存入 earthquake_list
for eq_id in uni_ids:
    if eq_id not in earthquake_list:
        earthquake_list.append(eq_id)

print(f"處理完成，目前 earthquake_list 總數: {len(earthquake_list)}")


AttributeError: 'DataFrame' object has no attribute 'str'

In [9]:
df

,source_origin_time,source_depth_km,source_latitude_deg,source_longitude_deg,source_magnitude,source_magnitude_type,source_gap_deg,source_erz_km,source_erh_km,source_rms_s,...,trace_E_snr_db,trace_pga_perg,trace_pga_cmps2,trace_Z_pga_cmps2,trace_pgv_cmps,trace_Z_pgv_cmps,trace_event_number,trace_completeness,year,pga
0,2012-05-20 22:39:44.899999,2.53,23.72,121.54,3.11,ml,129,0.3,0.1,0.29,...,35.796,0.168,1.647,0.731,0.044,0.019,1,4,2012,1.429186
1,2012-05-20 22:39:44.899999,2.53,23.72,121.54,3.11,ml,129,0.3,0.1,0.29,...,45.738,0.325,3.185,1.181,0.101,0.029,1,4,2012,2.480711
2,2012-05-20 22:39:44.899999,2.53,23.72,121.54,3.11,ml,129,0.3,0.1,0.29,...,5.948,0.031,0.305,0.107,0.013,0.004,1,4,2012,0.299020
3,2012-05-20 22:39:44.899999,2.53,23.72,121.54,3.11,ml,129,0.3,0.1,0.29,...,5.586,0.030,0.291,0.096,0.012,0.009,1,4,2012,0.282824
4,2012-05-20 22:39:44.899999,2.53,23.72,121.54,3.11,ml,129,0.3,0.1,0.29,...,27.435,0.012,0.119,0.073,0.004,0.002,1,4,2012,0.115394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465458,2021-10-03 07:44:40.130000,6.98,24.78,122.24,4.41,ml,129,0.4,0.3,0.54,...,-8.077,0.001,0.012,0.006,0.002,0.001,1,4,2021,0.013722
465459,2021-10-03 07:44:40.130000,6.98,24.78,122.24,4.41,ml,129,0.4,0.3,0.54,...,-0.044,0.001,0.007,0.006,0.001,0.001,1,3,2021,0.007312
465460,2021-10-03 07:44:40.130000,6.98,24.78,122.24,4.41,ml,129,0.4,0.3,0.54,...,1.993,0.001,0.007,0.003,0.002,0.001,1,4,2021,0.007040
465461,2021-10-03 07:44:40.130000,6.98,24.78,122.24,4.41,ml,129,0.4,0.3,0.54,...,6.927,0.001,0.008,0.003,0.001,0.001,1,3,2021,0.008423
